# Milestone 1 — arXiv Crawling

## Objectives

This milestone focuses on collecting arXiv paper metadata,
retrieving paper sources, and evaluating the crawling pipeline.


# Part I — Milestone 1: Data Crawling

## 2. Methodology

The input corpus is defined only by the assigned ID ranges in `config/aXirPaperID`, not by topic or date sampling. `assigned_base_ids()` expands every row into base IDs (`YYMM` prefix plus a zero-padded five-digit suffix, e.g. `2603.10580`). It fails fast if a range crosses months, is reversed, or overlaps another range, which gives 25,000 unique base IDs for the five members (5,000 each).

| Stage | Source | Tool | Raw output (`data/raw/`) |
|---|---|---|---|
| Version discovery | `https://arxiv.org/abs/<id>` (submission history) | `requests` + `BeautifulSoup` | `arxiv_history_all_assignments.jsonl`, `arxiv_discovery_status_all.csv` |
| Paper metadata | arXiv API, `id_list` of versioned IDs | `arxiv.py` (`client.results(search)`) | `arxiv_metadata_all_assignments.jsonl`, `arxiv_metadata_manifest.csv` |
| Source download | `https://export.arxiv.org/src/<id>v<n>` | `requests`, `tarfile` | `sources/<id>/v<n>/`, `source_download_manifest.csv` |
| Reference metadata | Semantic Scholar Graph API `/references` | `requests` | `semantic_scholar/<id>.json`, `semantic_scholar_manifest.csv` |

### 2.1 Entry discovery and version coverage

**Implementation.** For each base ID, `discover_assigned_versions()` requests the abstract page and reads the `div.submission-history` block. It takes the highest `vN` found and generates `v1 … vN`, so every version is covered. The result is appended to `arxiv_history_all_assignments.jsonl` as `{base_id, versions}`. Metadata for all discovered versions is then fetched with the arXiv API in batches of 50 versioned IDs (`crawl_arxiv_metadata()`). OAI-PMH and the Kaggle snapshot were not used.

**Missing or invalid IDs.**
- HTTP 404 is recorded as `not_found` and is not retried on later runs.
- A page without a submission-history block, or without any version number, is recorded as `error`. We never assume such a paper has a single version. It is retried on the next run.
- Every attempt, with its status, number of versions, elapsed time and error message, is appended to `arxiv_discovery_status_all.csv`.

**Reuse of earlier checkpoints.** Before sending any request, `_load_histories()` loads every existing `arxiv_history_*.jsonl`, including per-member files from earlier standalone runs. Metadata checkpoints (`arxiv_metadata_*_all_versions.jsonl`) are merged into the consolidated JSONL without duplicate IDs and are marked `record_source = existing_metadata_checkpoint` in the manifest. IDs that already have a result are skipped, so every stage is resumable.

**Error handling / retry / rate limiting:**
- **Pacing.** A `RequestPacer` enforces at least 3.1 s between requests (arXiv asks for 3 s or more), and `arxiv.Client` uses `delay_seconds=3.1`. One connection is used at a time, with a descriptive `User-Agent`.
- **Retry.** `_get_with_retries()` makes up to 5 attempts with a 60 s timeout. It retries on HTTP 429/500/502/503/504 and on network errors. It honours the `Retry-After` header and otherwise uses exponential back-off (`2^(attempt-1)` seconds plus jitter, capped at 60 s).
- **Metadata batches.** Each batch of 50 IDs has up to 5 attempts, and only the IDs not yet returned are queried again. IDs still missing after the last attempt are written with status `error`, so nothing is lost silently.
- **Manifests.** All manifests are append-only CSVs with `fsync`. The latest row per key defines the current status.

### 2.2 Source download and storage

Each version is fetched individually from the versioned endpoint `https://export.arxiv.org/src/<id>v<n>` (the export mirror, never the main site), with the same pacing and retry rules. The response is streamed into a temporary file and opened with `tarfile`. Only `.tex` and `.bib` members are extracted. Figure binaries and all other files never reach the corpus, while figure commands stay untouched inside the TeX text.

**Safe extraction.** Members are skipped if they are not regular files, are symlinks or hard links, have absolute paths or `..` components, or are duplicates. Extraction goes to a staging directory first and is moved into place with `os.replace`, so a crash cannot leave a half-written version folder.

**Directory convention:** `data/raw/sources/<arxiv_id>/v<n>/`, keeping the archive's internal sub-folders (e.g. `data/raw/sources/2603.10580/v2/main.tex`).

**Missing or non-TeX source handling.** The publication is never dropped. The version folder is created and left empty, and the outcome is recorded in `source_download_manifest.csv`:

| `status` | Meaning |
|---|---|
| `success` | at least one `.tex` or `.bib` file extracted |
| `no_source` | HTTP 404, no source available for this version |
| `non_archive` | response is not a readable tar archive |
| `no_tex_or_bib` | valid archive without `.tex` or `.bib` files (e.g. PDF-only) |
| `error` | network or file-system failure after retries (retried on the next run) |

The manifest also stores `archive_bytes`, `source_bytes_before_filter` and `source_bytes`, which are used to compare the size before and after removing figures.

**`.bbl` files.** The current implementation keeps only `.tex` and `.bib`, so a bibliography that exists only as a `.bbl` file is not retained. TODO: decide whether to keep `.bbl` for papers without `.bib` and without `\bibitem` (the course Q&A suggests keeping it), and report how many versions are affected.

### 2.3 Metadata and references

**Paper metadata** (arXiv API, one record per version): id, title, authors, abstract, published, updated, categories, primary category, comment, `journal_ref`, doi and pdf url. At table-building time (`build_tables.py`), `papers` takes title and authors from the latest version. `submission_date` is the earliest `published` date, `revised_dates` are the distinct `updated` dates different from it, and `venue` is the latest non-empty `journal_ref`.

**References** come from `https://api.semanticscholar.org/graph/v1/paper/arXiv:<id>/references`. We request the flat fields `citedPaper.paperId`, `title`, `authors`, `year`, `publicationDate`, `venue` and `externalIds`. The endpoint is paginated (`limit=1000`, `offset`), so papers with more than one page are not truncated.

- **Rate limiting.** Requests are paced at 3.1 s, and an API key can be supplied through the `S2_API_KEY` environment variable (sent as `x-api-key`).
- **Missing results.** HTTP 404 is stored as `not_found` with an empty reference list. Other failures are stored as `error` and retried on the next run.
- **Retry.** The same `_get_with_retries()` logic as in 2.1, including `Retry-After` and back-off.
- **Storage.** All returned references are saved to `semantic_scholar/<id>.json`, including those without an arXiv ID. Only references with `externalIds.ArXiv` become rows of the published `references` table. `ref_submission_date` is filled from S2 `publicationDate` (accepted by the instructors, since S2 has no submission date), and `ref_s2_id` from `paperId`.
- **Manifest.** `semantic_scholar_manifest.csv` records `reference_count` and `arxiv_reference_count` per paper.

All per-stage manifests are consolidated into `data/processed/crawl_manifest.csv` (one row per paper version), which is the basis of the coverage statistics in section 3.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))
from src.utils.table_io import load_table, PROCESSED_DIR

## 3. Milestone 1 — output tables and data quality

The final dataset tables introduced by Milestone 1 are `papers` and `references`. These tables are described here for analysis; the working corpus may contain additional fields/files.

### 3.1 Expected schema

**`papers`**

| Column | Meaning |
|---|---|
| `arxiv_id` | Base arXiv ID; required |
| `title` | Paper title |
| `authors` | List of author names |
| `submission_date` | Initial submission date |
| `revised_dates` | List of revision dates |
| `venue` | Venue when available; nullable |

**`references`**

| Column | Meaning |
|---|---|
| `arxiv_id` | Citing paper's arXiv ID |
| `ref_arxiv_id` | Referenced paper's arXiv ID |
| `ref_title` | Reference title |
| `ref_authors` | List of reference authors |
| `ref_submission_date` | Reference submission date |
| `ref_s2_id` | Semantic Scholar paper ID |

The working corpus should retain all crawled reference records. The published `references` table includes references with an arXiv ID, as required by the specification.

In [ ]:
PAPERS_PATH = PROCESSED_DIR / "papers.parquet"
REFERENCES_PATH = PROCESSED_DIR / "references.parquet"

papers = load_table(PAPERS_PATH)
references = load_table(REFERENCES_PATH)

print("papers:", papers.shape)
print("references:", references.shape)
display(papers.head())
display(references.head())

### 3.2 Crawling coverage and success rate

Report at least:
- Number of assigned base IDs attempted
- Number of papers resolved successfully
- Number of versions expected and number downloaded successfully
- Number of papers/versions with missing or non-TeX sources
- Overall success rate, with the denominator clearly defined

**Definition used for each rate:** TODO. Keep paper-level and version-level rates separate.

In [ ]:
MANIFEST_PATH = PROCESSED_DIR / "crawl_manifest.csv"
if MANIFEST_PATH.exists():
    crawl_manifest = pd.read_csv(MANIFEST_PATH)
    display(crawl_manifest.head())
    print("Manifest rows:", len(crawl_manifest))
else:
    crawl_manifest = pd.DataFrame()
    print(f"Create the crawler manifest at {MANIFEST_PATH} to calculate coverage.")



### 3.3 Corpus statistics and figures

Generate figures from the actual data during book compilation. Suggested analyses:
- Number of versions per publication
- Distribution of reference counts per paper
- Missingness of key metadata fields
- Average source size before and after excluding figure binaries, if both measurements were recorded

Interpret each figure in text; do not leave charts without an explanation.

In [ ]:
if not papers.empty:
    print("Paper table columns:", papers.columns.tolist())
if not references.empty:
    print("Reference table columns:", references.columns.tolist())

# Example plotting template (uncomment/adapt after confirming the real column names).
# if "ref_count" in papers.columns:
#     ax = papers["ref_count"].dropna().plot(kind="hist", bins=30)
#     ax.set_title("Distribution of references per paper")
#     ax.set_xlabel("References per paper")
#     ax.set_ylabel("Number of papers")
#     plt.tight_layout()
#     plt.show()

## 4. Milestone 1 — performance benchmark

Benchmark on the specified CPU-only Google Colab testbed. Explain whether timings are end-to-end wall time and how RAM/storage are measured.

Required report items:
- Total time for entry discovery
- Total end-to-end time to process the corpus
- Average processing time per required paper (define the denominator)
- Peak RAM and average RAM consumption
- Maximum disk storage required and final output storage size
- Average source/archive size before and after figure binaries are excluded
- Average references per paper and Semantic Scholar reference-metadata success rate

| Metric | Value | Measurement method / notes |
|---|---:|---|
| Entry discovery total time (s) | TODO | TODO |
| End-to-end crawling time (s) | TODO | TODO |
| Average time per paper (s) | TODO | TODO |
| Peak RAM (MiB/GiB) | TODO | TODO |
| Average RAM (MiB/GiB) | TODO | TODO |
| Maximum disk usage (MiB/GiB) | TODO | TODO |
| Final output size (MiB/GiB) | TODO | TODO |
| Average source size before filtering | TODO | TODO |
| Average source size after filtering | TODO | TODO |

### 4.1 Benchmark method

**Runtime measurement:** TODO  
**RAM measurement:** TODO  
**Disk usage measurement:** TODO  
**Hardware/runtime configuration:** TODO

## 5. Milestone 1 — discussion and limitations

Discuss:
- Which failure cases occurred most often?
- Were some versions unavailable or not provided as TeX?
- How complete was Semantic Scholar metadata?
- What trade-offs were made between request rate, runtime, retries, and reliability?
- What should be improved before parsing?

**Findings:** TODO

**Limitations:** TODO